In [1]:
!pip install datasets


Load data → wrap it as Documents → store it → embed it → run a pipeline → evaluate the answers → write/save docs 

Data comes from somewhere → load_dataset
Convert into Haystack format → Document
We’ll type-check lists → List
We will build an assembly line → Pipeline
We will create embeddings → SentenceTransformersDocumentEmbedder
We will evaluate quality → FaithfulnessEvaluator, SASEvaluator
We will store documents → DocumentWriter
Store is local/temporary → InMemoryDocumentStore
Handle duplicates → DuplicatePolicy

In [2]:
from datasets import load_dataset
from haystack import Document
from typing import List
from haystack import Pipeline
from haystack.components.embedders import SentenceTransformersDocumentEmbedder
from haystack.components.evaluators import FaithfulnessEvaluator, SASEvaluator
from haystack.components.writers import DocumentWriter
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.document_stores.types import DuplicatePolicy


d:\AI\KrishNaik_Academy\Coding\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()  # loads .env into environment variables

os.environ["OPENAI_API_KEY"]= os.getenv('OPENAI_API_KEY')

llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0.2
)

### 1) Load dataset + prepare lists

In [3]:
dataset=load_dataset("vblagoje/PubMedQA_instruction",split="train")
dataset=dataset.select(range(25))

# A RAG evaluation needs (question, context, ground-truth answer).
# You’re structuring the dataset into the exact shapes your pipeline/evaluators expect.

all_documents = [Document(content=doc["context"]) for doc in dataset]
all_questions = [doc["instruction"] for doc in dataset] 
all_ground_truth_answers = [doc["response"] for doc in dataset]


### 2) Create store + indexing components 

In [4]:
# document_embedder: converts each Document’s text into an embedding vector.
# document_writer: inserts documents into the store.
# DuplicatePolicy.SKIP: if duplicates appear, ignore them 
# A RAM-based store for documents + embeddings 

document_store = InMemoryDocumentStore()

document_embedder = SentenceTransformersDocumentEmbedder(model="sentence-transformers/all-MiniLM-L6-v2")
document_writer = DocumentWriter(document_store=document_store, policy=DuplicatePolicy.SKIP)

In [5]:
#3) Build and run the indexing pipeline
# documents → embedder → writer → store 

In [6]:
indexing=Pipeline()
indexing.add_component(instance = document_embedder, name = "document_embedder") 
indexing.add_component(instance = document_writer, name = "document_writer")  
indexing.connect("document_embedder.documents", "document_writer.documents")

# output named documents coming out of document_embedder 
# input named documents expected by document_writer

indexing.run({"document_embedder": {"documents": all_documents}}) 
# The document_store contains documents with embeddings.

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 359.27it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 1/1 [00:02<00:00,  2.73s/it]


{'document_writer': {'documents_written': 25}}

# 4) Import RAG components

In [7]:
from haystack.components.builders import AnswerBuilder, PromptBuilder
from haystack.components.embedders import SentenceTransformersTextEmbedder
from haystack.components.generators import OpenAIGenerator
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever

#SentenceTransformersTextEmbedder: embeds the query
#InMemoryEmbeddingRetriever: retrieves top_k docs using vector similarity
#PromptBuilder: stuffs docs + question into a prompt template
#OpenAIGenerator: LLM call to generate answer
#AnswerBuilder: packages replies + docs into “answers” objects

Embed query → Retrieve docs → Build prompt → Generate → Package answer

### 5) Prompt template (Jinja-style)

In [ ]:
template = """
        You have to answer the following question based on the given context information only.

        Context:
        {% for document in documents %}
            {{ document.content }}
        {% endfor %}

        Question: {{question}}
        Answer:
        """

# RAG works by forcing the model to answer using retrieved evidence.
# This phrase is your guardrail: “based on context information only

@ 6) Build the RAG pipeline

In [9]:
rag_pipeline = Pipeline()
#Embeds the user question into a vector.
rag_pipeline.add_component(
    "query_embedder", SentenceTransformersTextEmbedder(model="sentence-transformers/all-MiniLM-L6-v2")
)
#Retrieve 3 most similar docs from the store
rag_pipeline.add_component("retriever", InMemoryEmbeddingRetriever(document_store, top_k=3)) 

rag_pipeline.add_component("prompt_builder", PromptBuilder(template=template))
rag_pipeline.add_component("generator", OpenAIGenerator(model="gpt-3.5-turbo"))
rag_pipeline.add_component("answer_builder", AnswerBuilder())

# prompt_builder: merges retrieved docs + question into the prompt text
# generator: calls LLM
# answer_builder: collects final answer + docs for output

PromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.


### 7) Connect the RAG pipeline (the wiring) 




In [16]:
# 1) Output of query_embedder ->  retriever’s query_embedding [Retriever needs the query vector, not raw text] 
rag_pipeline.connect("query_embedder", "retriever.query_embedding")

#  Retriever outputs documents → prompt_builder consumes 
rag_pipeline.connect("retriever", "prompt_builder.documents")

# Prompt text → LLM. 
rag_pipeline.connect("prompt_builder", "generator")

# answers[0].data (the answer text)
# answers[0].documents (evidence docs used)
rag_pipeline.connect("generator.replies", "answer_builder.replies")
rag_pipeline.connect("retriever", "answer_builder.documents") 

🚅 Components
  - query_embedder: SentenceTransformersTextEmbedder
  - retriever: InMemoryEmbeddingRetriever
  - prompt_builder: PromptBuilder
  - generator: OpenAIGenerator
  - answer_builder: AnswerBuilder
🛤️ Connections
  - query_embedder.embedding -> retriever.query_embedding (list[float])
  - retriever.documents -> prompt_builder.documents (list[Document])
  - retriever.documents -> answer_builder.documents (list[Document])
  - prompt_builder.prompt -> generator.prompt (str)
  - generator.replies -> answer_builder.replies (list[str])

### Random sampling for evaluation 

In [17]:
import random

questions, ground_truth_answers, ground_truth_docs = zip(
    *random.sample(list(zip(all_questions, all_ground_truth_answers, all_documents)), 25)
)

# Zips the aligned lists into tuples
# samples 25 items randomly
# unzips back into 3 sequences

###  Run RAG for each question 

In [18]:
rag_answers = []
retrieved_docs = [] 
# Stores predictions + evidence. 

In [19]:
for question in list(questions):
    response = rag_pipeline.run(
        {
            "query_embedder": {"text": question},
            "prompt_builder": {"question": question},
            "answer_builder": {"query": question},
        }
    )
    rag_answers.append(response["answer_builder"]["answers"][0].data)
    retrieved_docs.append(response["answer_builder"]["answers"][0].documents)


Batches: 100%|██████████| 1/1 [00:00<00:00, 18.92it/s]


### Build evaluation pipeline 

In [21]:
eval_pipeline = Pipeline()
eval_pipeline.add_component("faithfulness", FaithfulnessEvaluator())
eval_pipeline.add_component("sas_evaluator", SASEvaluator(model="sentence-transformers/all-MiniLM-L6-v2"))

# faithfulness: “did you use context?”
# SAS: “does your answer match ground truth?”

PromptBuilder has 3 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.


###  Run evaluation 

In [22]:
results = eval_pipeline.run(
    {
        "faithfulness": {
            "questions": list(questions),
            "contexts": list([d.content] for d in ground_truth_docs),
            "predicted_answers": rag_answers,
        },
        "sas_evaluator": {"predicted_answers": rag_answers, "ground_truth_answers": list(ground_truth_answers)},
    }
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 285.29it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
100%|██████████| 25/25 [06:14<00:00, 14.96s/it]


### Wrap results nicely and print score report

In [23]:
from haystack.evaluation.eval_run_result import EvaluationRunResult # A helper class to package inputs + results.

inputs = {
    "question": list(questions),
    "contexts": list([d.content] for d in ground_truth_docs),
    "answer": list(ground_truth_answers),
    "predicted_answer": rag_answers,
}

In [26]:
from haystack.evaluation.eval_run_result import EvaluationRunResult
evaluation_result = EvaluationRunResult(run_name="pubmed_rag_pipeline", inputs=inputs, results=results)
print(evaluation_result.aggregated_report())

# Creates a named evaluation run and prints a summary report. 

{'metrics': ['faithfulness', 'sas_evaluator'], 'score': [np.float64(0.8856666666666666), np.float64(0.7105098164081574)]}
